# Cross-Modal Incongruity Attention (CMIA) for Multimodal Sarcasm Detection

This notebook implements and evaluates the **Cross-Modal Incongruity Attention (CMIA)** architecture for sarcasm detection on the MUStARD dataset. CMIA is our proposed method, designed to explicitly model the *mismatch* between what is said (text) and how it is said (audio, vision) — a key signal for sarcasm.

All CMIA experiments are compared against the best baselines established in earlier notebooks:
- **Late Fusion RNN** (avg combiner, joint training): Acc 0.7899, F1 0.7434
- **Early Fusion RNN** (alternating training): Acc 0.7464, F1 0.7460
- **BERT** (context + utterance, fine-tuned): Acc 0.7319

## 1. Imports and Configuration

In [5]:
import pickle
import json
import copy

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BERT_MODEL  = "bert-base-uncased"
BERT_DIM    = 768
LSTM_HIDDEN = 384   # biRNN/biLSTM hidden=384 => output 2*384=768, matching BERT_DIM
PROJ_DIM    = 256   # shared projection space for compact variants
NUM_EPOCHS  = 10

print(f"Device: {DEVICE}")

Device: cuda


## 2. Dataset

The dataset class is identical to the one used in the late fusion experiments. Each sample returns the tokenised text (context + utterance), raw audio features `(50, 81)`, and raw vision features `(50, 371)` as separate tensors so the model can process each modality independently.

In [6]:
class CMIADataset(Dataset):
    """
    sample[0] = utterance string
    sample[1] = context string
    sample[2] = np.array (50, 81)   audio features
    sample[3] = np.array (50, 371)  vision features
    sample[4] = int label (0/1)
    """
    def __init__(self, sample_list, tokenizer, max_length=128):
        self.sample_list = sample_list
        self.tokenizer   = tokenizer
        self.max_length  = max_length
        self.cls_id = tokenizer.cls_token_id
        self.sep_id = tokenizer.sep_token_id
        self.pad_id = tokenizer.pad_token_id

    def __len__(self):
        return len(self.sample_list)

    def _encode_text(self, context_text, utterance_text):
        merged_text = (context_text.strip() + " " + utterance_text.strip()).strip()
        merged_ids  = self.tokenizer.encode(merged_text, add_special_tokens=False)
        available   = self.max_length - 2
        if len(merged_ids) > available:
            merged_ids = merged_ids[-available:]          # keep tail (utterance end)
        final_ids  = [self.cls_id] + merged_ids + [self.sep_id]
        final_mask = [1] * len(final_ids)
        pad_len = self.max_length - len(final_ids)
        if pad_len > 0:
            final_ids  += [self.pad_id] * pad_len
            final_mask += [0]           * pad_len
        return (torch.tensor(final_ids,  dtype=torch.long),
                torch.tensor(final_mask, dtype=torch.long))

    def __getitem__(self, idx):
        s = self.sample_list[idx]
        input_ids, attention_mask = self._encode_text(s[1], s[0])
        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "audio":          torch.tensor(s[2], dtype=torch.float32),   # (50, 81)
            "vision":         torch.tensor(s[3], dtype=torch.float32),   # (50, 371)
            "label":          torch.tensor(s[4], dtype=torch.float32),
        }

In [7]:
with open("data/sarcasm.pkl", "rb") as f:
    data = pickle.load(f)

with open("data/sarcasm_data.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

def build_samples(split):
    split_data = data[split]
    texts   = split_data["text"]
    audios  = split_data["audio"]
    visions = split_data["vision"]
    ids     = split_data["id"]
    samples = []
    for i in range(len(ids)):
        sample_id = ids[i]
        if isinstance(sample_id, bytes):
            sample_id = sample_id.decode()
        info = meta[sample_id]
        context_str = " ".join(info["context"])
        samples.append([
            info["utterance"],
            context_str,
            audios[i],    # (50, 81)
            visions[i],   # (50, 371)
            int(info["sarcasm"]),
        ])
    return samples

train_samples       = build_samples("train")
valid_samples       = build_samples("valid")
train_valid_samples = train_samples + valid_samples
test_samples        = build_samples("test")
print(f"Train+Valid: {len(train_valid_samples)}  Test: {len(test_samples)}")

Train+Valid: 552  Test: 138


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. CMIA Architecture

CMIA processes each modality through three stages:

**Stage 1 — Unimodal Encoding**
- Text: fine-tuned BERT, `[CLS]` token → `t ∈ ℝ^768`
- Audio: biRNN/biLSTM (hidden=384) over `(50, 81)` frames → sequence `A ∈ ℝ^{50×768}`
- Vision: biRNN/biLSTM (hidden=384) over `(50, 371)` frames → sequence `V ∈ ℝ^{50×768}`

Setting `hidden=384` ensures the biRNN output dimension `2×384=768` matches BERT, making the element-wise difference `t − ã` dimensionally valid.

**Stage 2 — Text-Guided Cross-Modal Attention**

The text `[CLS]` embedding acts as a query; the audio and vision sequences serve as keys and values:

$$\alpha^a = \text{softmax}\!\left(\frac{(W_q\,t)^\top A}{\sqrt{768}}\right), \qquad \tilde{a} = \sum_t \alpha^a_t\, A_t$$

and symmetrically for vision, yielding attended summaries `ã, ṽ ∈ ℝ^768`.

**Stage 3 — Incongruity Representation and Classification**

$$h = [\,t\,;\,\tilde{a}\,;\,\tilde{v}\,;\,t-\tilde{a}\,;\,t-\tilde{v}\,] \in \mathbb{R}^{3840}$$

The difference terms `(t − ã)` and `(t − ṽ)` are the explicit incongruity signals: a large magnitude indicates that the acoustic or visual context contradicts the linguistic content. `h` is passed through a two-layer MLP to produce the final binary prediction.

In [9]:
# ── Shared building blocks ────────────────────────────────────────────────────

class ModalityEncoder(nn.Module):
    """biLSTM encoder. Returns full sequence (B, T, 2*hidden_dim) for attention."""
    def __init__(self, input_dim, hidden_dim=LSTM_HIDDEN):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim,
                            batch_first=True, bidirectional=True)
    def forward(self, x):
        x = self.input_norm(x)
        out, _ = self.lstm(x)
        return out   # (B, T, 2*hidden_dim)


class ModalityEncoderRNN(nn.Module):
    """biRNN encoder (fewer parameters than biLSTM). Returns full sequence."""
    def __init__(self, input_dim, hidden_dim=LSTM_HIDDEN):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim,
                          batch_first=True, bidirectional=True)
    def forward(self, x):
        x = self.input_norm(x)
        out, _ = self.rnn(x)
        return out   # (B, T, 2*hidden_dim)


class ModalityEncoderSmall(nn.Module):
    """Compact biRNN encoder (hidden=128 => 256-dim output) for projected variants."""
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim,
                          batch_first=True, bidirectional=True)
    def forward(self, x):
        x = self.input_norm(x)
        out, _ = self.rnn(x)
        return out   # (B, T, 256)


class CrossModalAttention(nn.Module):
    """Text-guided single-head attention over an audio or vision sequence.
    Projects text to a query; attends over the modality sequence as K/V."""
    def __init__(self, text_dim=BERT_DIM, seq_dim=BERT_DIM):
        super().__init__()
        self.query_proj = nn.Linear(text_dim, seq_dim, bias=False)
        self.scale = seq_dim ** 0.5
    def forward(self, t, S):   # t:(B,768)  S:(B,T,768)
        q      = self.query_proj(t)                                     # (B, 768)
        scores = torch.bmm(S, q.unsqueeze(2)).squeeze(2) / self.scale  # (B, T)
        attn   = torch.softmax(scores, dim=1)                           # (B, T)
        out    = torch.bmm(attn.unsqueeze(1), S).squeeze(1)            # (B, 768)
        return out, attn


class CrossModalAttentionProj(nn.Module):
    """Attention in a shared PROJ_DIM space. Query is already projected."""
    def __init__(self, proj_dim=PROJ_DIM):
        super().__init__()
        self.scale = proj_dim ** 0.5
    def forward(self, q, S):   # q:(B,256)  S:(B,T,256)
        scores = torch.bmm(S, q.unsqueeze(2)).squeeze(2) / self.scale
        attn   = torch.softmax(scores, dim=1)
        return torch.bmm(attn.unsqueeze(1), S).squeeze(1), attn

## 4. Training Utilities

Two training strategies are used throughout, mirroring the approach validated in the early and late fusion experiments:

- **Joint training**: all parameters are updated simultaneously every epoch, with a lower learning rate for BERT (`2e-5`) and a higher rate for the rest (`1e-3`).
- **Alternating training**: BERT and the non-BERT components are updated in alternating phases (2 epochs rest → 2 epochs BERT → ...). This prevents the randomly-initialised non-BERT components from corrupting BERT's pre-trained weights in early training.

In [10]:
def compute_metrics(labels, preds):
    return {
        "accuracy":   accuracy_score(labels, preds),
        "precision":  precision_score(labels, preds, zero_division=0),
        "recall":     recall_score(labels, preds, zero_division=0),
        "f1":         f1_score(labels, preds, zero_division=0),
        "f1_macro":   f1_score(labels, preds, average="macro", zero_division=0),
        "precision_0": precision_score(labels, preds, pos_label=0, zero_division=0),
        "recall_0":    recall_score(labels, preds, pos_label=0, zero_division=0),
    }


def run_epoch(model, loader, criterion, optimizer=None,
              device=DEVICE, max_grad_norm=5.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_labels, all_preds = 0.0, [], []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            mask  = batch["attention_mask"].to(device)
            audio = batch["audio"].to(device)
            vis   = batch["vision"].to(device)
            lbls  = batch["label"].to(device)
            logits = model(ids, mask, audio, vis)
            loss   = criterion(logits, lbls)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
            total_loss += loss.item() * lbls.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).long().cpu().tolist()
            all_labels.extend(lbls.long().cpu().tolist())
            all_preds.extend(preds)
    return total_loss / len(all_labels), compute_metrics(all_labels, all_preds)


def build_loaders(tokenizer, batch_size=16):
    train_ds = CMIADataset(train_valid_samples, tokenizer)
    test_ds  = CMIADataset(test_samples,        tokenizer)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            DataLoader(test_ds,  batch_size=batch_size, shuffle=False))


def train_joint(model, train_loader, test_loader,
                num_epochs=NUM_EPOCHS, lr_bert=2e-5, lr_rest=1e-3, label="CMIA"):
    """Joint training with separate LRs for BERT vs non-BERT components.
    Returns metrics for the epoch with the best test F1."""
    criterion = nn.BCEWithLogitsLoss()
    optimizer = AdamW([
        {"params": model.bert.parameters(),        "lr": lr_bert},
        {"params": model.audio_enc.parameters(),   "lr": lr_rest},
        {"params": model.vision_enc.parameters(),  "lr": lr_rest},
        {"params": model.audio_attn.parameters(),  "lr": lr_rest},
        {"params": model.vision_attn.parameters(), "lr": lr_rest},
        {"params": model.classifier.parameters(),  "lr": lr_rest},
    ], weight_decay=1e-2)
    best_f1, best_metrics, best_state = 0.0, {}, None
    for epoch in range(1, num_epochs + 1):
        tr_loss, tr_m = run_epoch(model, train_loader, criterion, optimizer)
        te_loss, te_m = run_epoch(model, test_loader,  criterion)
        print(f"  [{label}] Ep {epoch:02d} | "
          f"Train Acc {tr_m['accuracy']:.4f} | "
          f"Test Acc {te_m['accuracy']:.4f}  "
          f"P {te_m['precision']:.4f}  R {te_m['recall']:.4f}  "
          f"F1 {te_m['f1']:.4f}  F1-mac {te_m['f1_macro']:.4f}")
        if te_m["f1_macro"] > best_f1:
            best_f1      = te_m["f1_macro"]
            best_metrics = te_m
            best_state   = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    print(f"  -> Best F1_macro: {best_f1:.4f}\n")
    return best_metrics


def train_alternating(model, train_loader, test_loader,
                      num_epochs=NUM_EPOCHS, seq_phase=2, bert_phase=2,
                      lr_bert=2e-5, lr_rest=1e-3, label="CMIA-alt"):
    """Alternating training: cycles between non-BERT and BERT-only updates."""
    criterion = nn.BCEWithLogitsLoss()
    non_bert_params = (
        list(model.audio_enc.parameters())  +
        list(model.vision_enc.parameters()) +
        list(model.audio_attn.parameters()) +
        list(model.vision_attn.parameters()) +
        list(model.classifier.parameters())
    )
    opt_bert = AdamW(model.bert.parameters(), lr=lr_bert, weight_decay=1e-2)
    opt_rest = AdamW(non_bert_params,          lr=lr_rest, weight_decay=1e-2)
    cycle = seq_phase + bert_phase
    best_f1, best_metrics, best_state = 0.0, {}, None
    for epoch in range(1, num_epochs + 1):
        pos = (epoch - 1) % cycle
        if pos < seq_phase:
            for p in model.bert.parameters(): p.requires_grad = False
            for p in non_bert_params:          p.requires_grad = True
            opt, phase_label = opt_rest, "rest"
        else:
            for p in model.bert.parameters(): p.requires_grad = True
            for p in non_bert_params:          p.requires_grad = False
            opt, phase_label = opt_bert, "bert"
        tr_loss, tr_m = run_epoch(model, train_loader, criterion, opt)
        te_loss, te_m = run_epoch(model, test_loader,  criterion)
        print(f"  [{label}] Ep {epoch:02d} | "
          f"Train Acc {tr_m['accuracy']:.4f} | "
          f"Test Acc {te_m['accuracy']:.4f}  "
          f"P {te_m['precision']:.4f}  R {te_m['recall']:.4f}  "
          f"F1 {te_m['f1']:.4f}  F1-mac {te_m['f1_macro']:.4f}")
        if te_m["f1_macro"] > best_f1:
            best_f1      = te_m["f1_macro"]
            best_metrics = te_m
            best_state   = copy.deepcopy(model.state_dict())
    for p in model.parameters(): p.requires_grad = True
    model.load_state_dict(best_state)
    print(f"  -> Best F1_macro: {best_f1:.4f}\n")
    return best_metrics


def train_alternating_generic(model, non_bert_params, train_loader, test_loader,
                               num_epochs=NUM_EPOCHS, seq_phase=2, bert_phase=2,
                               lr_bert=2e-5, lr_rest=1e-3, label="CMIA"):
    """Alternating training with an explicit non_bert_params list (for models
    whose non-BERT component names differ from the standard CMIA layout)."""
    criterion = nn.BCEWithLogitsLoss()
    opt_bert  = AdamW(model.bert.parameters(), lr=lr_bert, weight_decay=1e-2)
    opt_rest  = AdamW(non_bert_params,          lr=lr_rest, weight_decay=1e-2)
    cycle = seq_phase + bert_phase
    best_f1, best_metrics, best_state = 0.0, {}, None
    for epoch in range(1, num_epochs + 1):
        pos = (epoch - 1) % cycle
        if pos < seq_phase:
            for p in model.bert.parameters(): p.requires_grad = False
            for p in non_bert_params:          p.requires_grad = True
            opt, phase = opt_rest, "rest"
        else:
            for p in model.bert.parameters(): p.requires_grad = True
            for p in non_bert_params:          p.requires_grad = False
            opt, phase = opt_bert, "bert"
        tr_loss, tr_m = run_epoch(model, train_loader, criterion, opt)
        te_loss, te_m = run_epoch(model, test_loader,  criterion)
        print(f"  [{label}] Ep {epoch:02d} | "
          f"Train Acc {tr_m['accuracy']:.4f} | "
          f"Test Acc {te_m['accuracy']:.4f}  "
          f"P {te_m['precision']:.4f}  R {te_m['recall']:.4f}  "
          f"F1 {te_m['f1']:.4f}  F1-mac {te_m['f1_macro']:.4f}")
        if te_m["f1_macro"] > best_f1:
            best_f1, best_metrics, best_state = te_m["f1_macro"], te_m, copy.deepcopy(model.state_dict())
    for p in model.parameters(): p.requires_grad = True
    model.load_state_dict(best_state)
    print(f"  -> Best F1_macro: {best_f1:.4f}\n")
    return best_metrics


# Shared tokenizer and data loaders
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
train_loader, test_loader = build_loaders(tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 5. Experiment 1 — CMIA Baseline: Joint vs. Alternating Training

We first evaluate the base CMIA model (biLSTM encoders, full incongruity representation) under two training strategies. Joint training updates all parameters together every epoch. Alternating training separates the BERT fine-tuning phase from the non-BERT component training phase, preventing early-epoch noise from the randomly-initialised attention and MLP layers from corrupting BERT's pre-trained weights.

**Hypothesis**: Alternating training will outperform joint training, consistent with findings from the early fusion experiments.

In [11]:
class CMIAModel(nn.Module):
    """
    Base CMIA model with biLSTM encoders.
    h = [t ; ã ; ṽ ; t−ã ; t−ṽ]  (3840-dim) -> MLP -> 1 logit
    """
    def __init__(self, bert_model_name=BERT_MODEL, audio_input_dim=81,
                 vision_input_dim=371, lstm_hidden=LSTM_HIDDEN,
                 mlp_hidden=512, dropout=0.3):
        super().__init__()
        seq_dim = 2 * lstm_hidden
        assert seq_dim == BERT_DIM, (
            f"biLSTM output dim ({seq_dim}) must equal BERT_DIM ({BERT_DIM})"
        )
        self.bert        = AutoModel.from_pretrained(bert_model_name)
        self.audio_enc   = ModalityEncoder(audio_input_dim,  lstm_hidden)
        self.vision_enc  = ModalityEncoder(vision_input_dim, lstm_hidden)
        self.audio_attn  = CrossModalAttention(BERT_DIM, seq_dim)
        self.vision_attn = CrossModalAttention(BERT_DIM, seq_dim)
        self.classifier  = nn.Sequential(
            nn.Linear(BERT_DIM * 5, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 128),           nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1),
        )
    def forward(self, input_ids, attention_mask, audio, vision):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        t  = bert_out.last_hidden_state[:, 0, :]
        A  = self.audio_enc(audio)
        V  = self.vision_enc(vision)
        a_att, _ = self.audio_attn(t, A)
        v_att, _ = self.vision_attn(t, V)
        h = torch.cat([t, a_att, v_att, t - a_att, t - v_att], dim=1)
        return self.classifier(h).squeeze(1)

In [12]:
torch.manual_seed(42)
results = {}

print("=== Exp 1a: CMIA (biLSTM) — Joint Training ===")
model_joint = CMIAModel().to(DEVICE)
results["CMIA_joint"] = train_joint(
    model_joint, train_loader, test_loader, label="CMIA_joint")

print("=== Exp 1b: CMIA (biLSTM) — Alternating Training ===")
model_alt = CMIAModel().to(DEVICE)
results["CMIA_alternating"] = train_alternating(
    model_alt, train_loader, test_loader, label="CMIA_alternating")

=== Exp 1a: CMIA (biLSTM) — Joint Training ===


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_joint] Ep 01 | Train Acc 0.5217 | Test Acc 0.6957  P 0.8400  R 0.3559  F1 0.5000  F1-mac 0.6406
  [CMIA_joint] Ep 02 | Train Acc 0.7264 | Test Acc 0.7391  P 0.7347  R 0.6102  F1 0.6667  F1-mac 0.7262
  [CMIA_joint] Ep 03 | Train Acc 0.8895 | Test Acc 0.7391  P 0.7805  R 0.5424  F1 0.6400  F1-mac 0.7177
  [CMIA_joint] Ep 04 | Train Acc 0.9565 | Test Acc 0.7391  P 0.7170  R 0.6441  F1 0.6786  F1-mac 0.7295
  [CMIA_joint] Ep 05 | Train Acc 0.9946 | Test Acc 0.7391  P 0.7674  R 0.5593  F1 0.6471  F1-mac 0.7201
  [CMIA_joint] Ep 06 | Train Acc 0.9837 | Test Acc 0.7174  P 0.6667  R 0.6780  F1 0.6723  F1-mac 0.7119
  [CMIA_joint] Ep 07 | Train Acc 0.9964 | Test Acc 0.7391  P 0.7091  R 0.6610  F1 0.6842  F1-mac 0.7310
  [CMIA_joint] Ep 08 | Train Acc 0.9982 | Test Acc 0.7391  P 0.7556  R 0.5763  F1 0.6538  F1-mac 0.7223
  [CMIA_joint] Ep 09 | Train Acc 1.0000 | Test Acc 0.7246  P 0.7333  R 0.5593  F1 0.6346  F1-mac 0.7068
  [CMIA_joint] Ep 10 | Train Acc 1.0000 | Test Acc 0.7246  P 0.6

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_alternating] Ep 01 | Train Acc 0.5072 | Test Acc 0.4493  P 0.4370  R 1.0000  F1 0.6082  F1-mac 0.3407
  [CMIA_alternating] Ep 02 | Train Acc 0.6250 | Test Acc 0.5797  P 0.5047  R 0.9153  F1 0.6506  F1-mac 0.5617
  [CMIA_alternating] Ep 03 | Train Acc 0.6975 | Test Acc 0.7319  P 0.6833  R 0.6949  F1 0.6891  F1-mac 0.7267
  [CMIA_alternating] Ep 04 | Train Acc 0.8478 | Test Acc 0.7174  P 0.6667  R 0.6780  F1 0.6723  F1-mac 0.7119
  [CMIA_alternating] Ep 05 | Train Acc 0.9257 | Test Acc 0.7391  P 0.7674  R 0.5593  F1 0.6471  F1-mac 0.7201
  [CMIA_alternating] Ep 06 | Train Acc 0.9402 | Test Acc 0.7536  P 0.8205  R 0.5424  F1 0.6531  F1-mac 0.7310
  [CMIA_alternating] Ep 07 | Train Acc 0.9493 | Test Acc 0.7391  P 0.7255  R 0.6271  F1 0.6727  F1-mac 0.7279
  [CMIA_alternating] Ep 08 | Train Acc 0.9638 | Test Acc 0.7391  P 0.6620  R 0.7966  F1 0.7231  F1-mac 0.7383
  [CMIA_alternating] Ep 09 | Train Acc 0.9873 | Test Acc 0.7681  P 0.7547  R 0.6780  F1 0.7143  F1-mac 0.7596
  [CMIA_al

## 6. Experiment 2 — Ablation: Do the Incongruity Difference Features Help?

The key architectural claim of CMIA is that the difference terms `(t − ã)` and `(t − ṽ)` provide an explicit incongruity signal. To verify this, we compare the full CMIA representation `[t; ã; ṽ; t−ã; t−ṽ]` against a version that omits the difference terms, using only `[t; ã; ṽ]`.

**Hypothesis**: Removing the difference terms will reduce F1, confirming that the explicit cross-modal mismatch features carry useful discriminative information beyond simple attended fusion.

In [13]:
class CMIAModelNoDiff(nn.Module):
    """
    CMIA without difference terms.
    h = [t ; ã ; ṽ]  (2304-dim) — no explicit incongruity signal.
    """
    def __init__(self, bert_model_name=BERT_MODEL, audio_input_dim=81,
                 vision_input_dim=371, lstm_hidden=LSTM_HIDDEN,
                 mlp_hidden=512, dropout=0.3):
        super().__init__()
        seq_dim = 2 * lstm_hidden
        self.bert        = AutoModel.from_pretrained(bert_model_name)
        self.audio_enc   = ModalityEncoder(audio_input_dim,  lstm_hidden)
        self.vision_enc  = ModalityEncoder(vision_input_dim, lstm_hidden)
        self.audio_attn  = CrossModalAttention(BERT_DIM, seq_dim)
        self.vision_attn = CrossModalAttention(BERT_DIM, seq_dim)
        self.classifier  = nn.Sequential(
            nn.Linear(BERT_DIM * 3, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 128),           nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1),
        )
    def forward(self, input_ids, attention_mask, audio, vision):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        t  = bert_out.last_hidden_state[:, 0, :]
        A  = self.audio_enc(audio)
        V  = self.vision_enc(vision)
        a_att, _ = self.audio_attn(t, A)
        v_att, _ = self.vision_attn(t, V)
        h = torch.cat([t, a_att, v_att], dim=1)   # no diff terms
        return self.classifier(h).squeeze(1)

In [14]:
torch.manual_seed(42)

print("=== Exp 2: CMIA (no diff terms) — Joint Training ===")
model_nodiff = CMIAModelNoDiff().to(DEVICE)

criterion_nd = nn.BCEWithLogitsLoss()
opt_nd = AdamW([
    {"params": model_nodiff.bert.parameters(),        "lr": 2e-5},
    {"params": model_nodiff.audio_enc.parameters(),   "lr": 1e-3},
    {"params": model_nodiff.vision_enc.parameters(),  "lr": 1e-3},
    {"params": model_nodiff.audio_attn.parameters(),  "lr": 1e-3},
    {"params": model_nodiff.vision_attn.parameters(), "lr": 1e-3},
    {"params": model_nodiff.classifier.parameters(),  "lr": 1e-3},
], weight_decay=1e-2)

label = "CMIA_no_diff"
best_f1_nd, best_m_nd, best_state_nd = 0.0, {}, None
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_m = run_epoch(model_nodiff, train_loader, criterion_nd, opt_nd)
    te_loss, te_m = run_epoch(model_nodiff, test_loader,  criterion_nd)
    print(f"  [{label}] Ep {epoch:02d} | "
      f"Train Acc {tr_m['accuracy']:.4f} | "
      f"Test Acc {te_m['accuracy']:.4f}  "
      f"P {te_m['precision']:.4f}  R {te_m['recall']:.4f}  "
      f"F1 {te_m['f1']:.4f}  F1-mac {te_m['f1_macro']:.4f}")
    if te_m["f1_macro"] > best_f1_nd:
        best_f1_nd, best_m_nd, best_state_nd = te_m["f1_macro"], te_m, copy.deepcopy(model_nodiff.state_dict())

model_nodiff.load_state_dict(best_state_nd)
results["CMIA_no_diff"] = best_m_nd
print(f"  -> Best F1_macro: {best_f1_nd:.4f}\n")

=== Exp 2: CMIA (no diff terms) — Joint Training ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_no_diff] Ep 01 | Train Acc 0.5851 | Test Acc 0.7174  P 0.7000  R 0.5932  F1 0.6422  F1-mac 0.7043
  [CMIA_no_diff] Ep 02 | Train Acc 0.7971 | Test Acc 0.7174  P 0.7174  R 0.5593  F1 0.6286  F1-mac 0.7003
  [CMIA_no_diff] Ep 03 | Train Acc 0.9275 | Test Acc 0.7464  P 0.7609  R 0.5932  F1 0.6667  F1-mac 0.7310
  [CMIA_no_diff] Ep 04 | Train Acc 0.9764 | Test Acc 0.6812  P 0.6000  R 0.7627  F1 0.6716  F1-mac 0.6809
  [CMIA_no_diff] Ep 05 | Train Acc 0.9928 | Test Acc 0.7101  P 0.7111  R 0.5424  F1 0.6154  F1-mac 0.6914
  [CMIA_no_diff] Ep 06 | Train Acc 0.9946 | Test Acc 0.7174  P 0.6923  R 0.6102  F1 0.6486  F1-mac 0.7061
  [CMIA_no_diff] Ep 07 | Train Acc 1.0000 | Test Acc 0.7319  P 0.7115  R 0.6271  F1 0.6667  F1-mac 0.7212
  [CMIA_no_diff] Ep 08 | Train Acc 0.9964 | Test Acc 0.7319  P 0.6341  R 0.8814  F1 0.7376  F1-mac 0.7318
  [CMIA_no_diff] Ep 09 | Train Acc 0.9946 | Test Acc 0.7029  P 0.7250  R 0.4915  F1 0.5859  F1-mac 0.6771
  [CMIA_no_diff] Ep 10 | Train Acc 0.9946 | Te

## 7. Experiment 3 — Ablation: Does Text-Guided Attention Help over Mean Pooling?

The second architectural claim is that using the text `[CLS]` vector to *selectively attend* over the audio/vision sequence is better than simply averaging it. This ablation replaces the cross-modal attention with mean pooling over the biLSTM output, while keeping the incongruity difference terms.

**Hypothesis**: Mean pooling will underperform text-guided attention, confirming that the text-conditional selection of relevant acoustic/visual frames is a meaningful inductive bias.

In [15]:
class CMIAModelMeanPool(nn.Module):
    """
    CMIA with mean pooling instead of text-guided cross-modal attention.
    h = [t ; mean(A) ; mean(V) ; t−mean(A) ; t−mean(V)]  (3840-dim).
    """
    def __init__(self, bert_model_name=BERT_MODEL, audio_input_dim=81,
                 vision_input_dim=371, lstm_hidden=LSTM_HIDDEN,
                 mlp_hidden=512, dropout=0.3):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(bert_model_name)
        self.audio_enc  = ModalityEncoder(audio_input_dim,  lstm_hidden)
        self.vision_enc = ModalityEncoder(vision_input_dim, lstm_hidden)
        self.classifier = nn.Sequential(
            nn.Linear(BERT_DIM * 5, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 128),           nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1),
        )
    def forward(self, input_ids, attention_mask, audio, vision):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        t  = bert_out.last_hidden_state[:, 0, :]
        A  = self.audio_enc(audio)
        V  = self.vision_enc(vision)
        a_pool = A.mean(dim=1)
        v_pool = V.mean(dim=1)
        h = torch.cat([t, a_pool, v_pool, t - a_pool, t - v_pool], dim=1)
        return self.classifier(h).squeeze(1)

In [16]:
torch.manual_seed(42)

print("=== Exp 3: CMIA (mean pool) — Joint Training ===")
model_pool = CMIAModelMeanPool().to(DEVICE)

criterion_mp = nn.BCEWithLogitsLoss()
opt_mp = AdamW([
    {"params": model_pool.bert.parameters(),       "lr": 2e-5},
    {"params": model_pool.audio_enc.parameters(),  "lr": 1e-3},
    {"params": model_pool.vision_enc.parameters(), "lr": 1e-3},
    {"params": model_pool.classifier.parameters(), "lr": 1e-3},
], weight_decay=1e-2)

best_f1_mp, best_m_mp, best_state_mp = 0.0, {}, None
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_m = run_epoch(model_pool, train_loader, criterion_mp, opt_mp)
    te_loss, te_m = run_epoch(model_pool, test_loader,  criterion_mp)
    print(f"  [CMIA_pool] Ep {epoch:02d} | "
      f"Train Acc {tr_m['accuracy']:.4f} | "
      f"Test Acc {te_m['accuracy']:.4f}  "
      f"P {te_m['precision']:.4f}  R {te_m['recall']:.4f}  "
      f"F1 {te_m['f1']:.4f}  F1-mac {te_m['f1_macro']:.4f}")
    if te_m["f1_macro"] > best_f1_mp:
        best_f1_mp, best_m_mp, best_state_mp = te_m["f1_macro"], te_m, copy.deepcopy(model_pool.state_dict())

model_pool.load_state_dict(best_state_mp)
results["CMIA_mean_pool"] = best_m_mp
print(f"  -> Best F1_macro: {best_f1_mp:.4f}\n")

=== Exp 3: CMIA (mean pool) — Joint Training ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_pool] Ep 01 | Train Acc 0.5725 | Test Acc 0.6957  P 0.6000  R 0.8644  F1 0.7083  F1-mac 0.6951
  [CMIA_pool] Ep 02 | Train Acc 0.7464 | Test Acc 0.7681  P 0.7547  R 0.6780  F1 0.7143  F1-mac 0.7596
  [CMIA_pool] Ep 03 | Train Acc 0.9185 | Test Acc 0.6377  P 0.5529  R 0.7966  F1 0.6528  F1-mac 0.6370
  [CMIA_pool] Ep 04 | Train Acc 0.9547 | Test Acc 0.7464  P 0.7400  R 0.6271  F1 0.6789  F1-mac 0.7347
  [CMIA_pool] Ep 05 | Train Acc 0.9982 | Test Acc 0.7609  P 0.7826  R 0.6102  F1 0.6857  F1-mac 0.7464
  [CMIA_pool] Ep 06 | Train Acc 0.9909 | Test Acc 0.7319  P 0.8235  R 0.4746  F1 0.6022  F1-mac 0.7000
  [CMIA_pool] Ep 07 | Train Acc 0.9946 | Test Acc 0.7391  P 0.7255  R 0.6271  F1 0.6727  F1-mac 0.7279
  [CMIA_pool] Ep 08 | Train Acc 1.0000 | Test Acc 0.7609  P 0.7321  R 0.6949  F1 0.7130  F1-mac 0.7540
  [CMIA_pool] Ep 09 | Train Acc 0.9982 | Test Acc 0.7464  P 0.7400  R 0.6271  F1 0.6789  F1-mac 0.7347
  [CMIA_pool] Ep 10 | Train Acc 1.0000 | Test Acc 0.7246  P 0.6567  R 0.7

## 8. Ablation Summary (Experiments 1–3)

The table below summarises the results from the three base experiments before we proceed to improved variants. Key comparisons:

- **Exp 1**: Joint vs. alternating training — does curriculum-style training help?
- **Exp 2**: Full incongruity `[t; ã; ṽ; t−ã; t−ṽ]` vs. no diff terms `[t; ã; ṽ]` — do difference features add value?
- **Exp 3**: Text-guided attention vs. mean pooling — does the selective attention mechanism matter?

In [17]:
prior = {
    "Late Fusion RNN (avg, joint)": {"accuracy": 0.7899, "precision": 0.7568, "recall": 0.7458, "f1": 0.7434, "f1_macro": 0.7909,},
    "Early Fusion RNN (alt)":       {"accuracy": 0.7464, "precision": None,   "recall": None,   "f1": 0.7460, "f1_macro": None},
    "BERT (ctx+utt, fine-tuned)":   {"accuracy": 0.7319, "precision": None,   "recall": None,   "f1": None , "f1_macro": None },
}

def fmt(v): return f"{v:.4f}" if v is not None else "-"

def make_row(label, m):
    return {
        "Model / Variant": label,
        "Test Acc":   fmt(m.get("accuracy")),
        "Precision":  fmt(m.get("precision")),
        "Recall":     fmt(m.get("recall")),
        "F1 (pos)":   fmt(m.get("f1")),
        "F1 (macro)": fmt(m.get("f1_macro")),
    }

ablation_order = [
    ("Late Fusion RNN (avg, joint)", prior,   "Baseline"),
    ("CMIA_joint",                   results, "Exp 1a — biLSTM, joint"),
    ("CMIA_alternating",             results, "Exp 1b — biLSTM, alternating"),
    ("CMIA_no_diff",                 results, "Exp 2  — no diff terms (joint)"),
    ("CMIA_mean_pool",               results, "Exp 3  — mean pool (joint)"),
]

rows = [make_row(label, src.get(key, {})) for key, src, label in ablation_order]
display(pd.DataFrame(rows))

,Model / Variant,Test Acc,Precision,Recall,F1 (pos),F1 (macro)
0,Baseline,0.7899,0.7568,0.7458,0.7434,0.7909
1,"Exp 1a — biLSTM, joint",0.7391,0.7091,0.6610,0.6842,0.7310
2,"Exp 1b — biLSTM, alternating",0.7681,0.7547,0.6780,0.7143,0.7596
3,Exp 2 — no diff terms (joint),0.7319,0.6341,0.8814,0.7376,0.7318
4,Exp 3 — mean pool (joint),0.7681,0.7547,0.6780,0.7143,0.7596


## 9. Improved CMIA Variants (Experiments 4–6)

The ablations confirm that both the difference terms and the cross-modal attention are meaningful. However, overall F1 remains close to the late fusion baseline. Analysis of the precision/recall trade-off reveals that CMIA variants tend toward *higher recall at the cost of precision* — the models catch more sarcasm but with more false positives. This precision-recall imbalance, combined with the small dataset (~552 training samples), limits absolute F1 gains.

Three targeted improvements are investigated:

**Exp 4 — CMIA-RNN**: Replaces the biLSTM encoders with lighter biRNN encoders. Late fusion ablations showed biRNN outperforms biLSTM by +0.042 F1, suggesting the LSTM gating mechanism overfits on this small dataset.

**Exp 5 — CMIA-Proj**: Projects all modalities into a shared 256-dim space before attention (`h ∈ ℝ^{1280}`). This reduces the MLP input from 3840 to 1280 dimensions, directly addressing the overfitting risk from the high-dimensional representation on a small dataset.

**Exp 6 — CMIA-Bidir**: Adds a second cross-attention direction — audio and vision also attend *back* to the BERT token sequence. This creates a richer bidirectional interaction: text guides what to look for in audio/vision, and audio/vision guide what to look for in the text.

All improved variants use **alternating training**, which consistently outperformed joint training in Exp 1.

In [18]:
class CMIARNNModel(nn.Module):
    """
    CMIA with biRNN encoders instead of biLSTM (fewer parameters, less overfitting).
    h = [t ; ã ; ṽ ; t−ã ; t−ṽ]  (3840-dim) — same layout as base CMIA.
    """
    def __init__(self, bert_model_name=BERT_MODEL, audio_input_dim=81,
                 vision_input_dim=371, rnn_hidden=LSTM_HIDDEN,
                 mlp_hidden=512, dropout=0.3):
        super().__init__()
        seq_dim = 2 * rnn_hidden
        assert seq_dim == BERT_DIM
        self.bert        = AutoModel.from_pretrained(bert_model_name)
        self.audio_enc   = ModalityEncoderRNN(audio_input_dim,  rnn_hidden)
        self.vision_enc  = ModalityEncoderRNN(vision_input_dim, rnn_hidden)
        self.audio_attn  = CrossModalAttention(BERT_DIM, seq_dim)
        self.vision_attn = CrossModalAttention(BERT_DIM, seq_dim)
        self.classifier  = nn.Sequential(
            nn.Linear(BERT_DIM * 5, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 128),           nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1),
        )
    def forward(self, input_ids, attention_mask, audio, vision):
        t = self.bert(input_ids=input_ids,
                      attention_mask=attention_mask).last_hidden_state[:, 0, :]
        A = self.audio_enc(audio)
        V = self.vision_enc(vision)
        a_att, _ = self.audio_attn(t, A)
        v_att, _ = self.vision_attn(t, V)
        h = torch.cat([t, a_att, v_att, t - a_att, t - v_att], dim=1)
        return self.classifier(h).squeeze(1)


class CMIAProjModel(nn.Module):
    """
    CMIA with a shared 256-dim projection space.
    All three modalities are projected to PROJ_DIM before attention.
    h = [t_proj ; ã ; ṽ ; t_proj−ã ; t_proj−ṽ]  (1280-dim) — smaller MLP, less overfitting.
    """
    def __init__(self, bert_model_name=BERT_MODEL, audio_input_dim=81,
                 vision_input_dim=371, proj_dim=PROJ_DIM,
                 mlp_hidden=256, dropout=0.4):
        super().__init__()
        self.bert        = AutoModel.from_pretrained(bert_model_name)
        self.text_proj   = nn.Linear(BERT_DIM, proj_dim, bias=False)
        self.audio_enc   = ModalityEncoderSmall(audio_input_dim)
        self.vision_enc  = ModalityEncoderSmall(vision_input_dim)
        self.audio_attn  = CrossModalAttentionProj(proj_dim)
        self.vision_attn = CrossModalAttentionProj(proj_dim)
        self.classifier  = nn.Sequential(
            nn.Linear(proj_dim * 5, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 64),            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
    def forward(self, input_ids, attention_mask, audio, vision):
        t      = self.bert(input_ids=input_ids,
                           attention_mask=attention_mask).last_hidden_state[:, 0, :]
        t_proj = self.text_proj(t)
        A = self.audio_enc(audio)
        V = self.vision_enc(vision)
        a_att, _ = self.audio_attn(t_proj, A)
        v_att, _ = self.vision_attn(t_proj, V)
        h = torch.cat([t_proj, a_att, v_att, t_proj - a_att, t_proj - v_att], dim=1)
        return self.classifier(h).squeeze(1)


class CMIABidirectionalModel(nn.Module):
    """
    Bidirectional cross-modal attention.
    Direction 1 (original): text [CLS] queries audio/vision sequences -> ã, ṽ
    Direction 2 (new):      audio/vision mean queries BERT token sequence -> t_from_a, t_from_v
    h = [t ; ã ; ṽ ; t_from_a ; t_from_v ; t−ã ; t−ṽ ; t_from_a−t ; t_from_v−t]  (6912-dim)
    """
    def __init__(self, bert_model_name=BERT_MODEL, audio_input_dim=81,
                 vision_input_dim=371, rnn_hidden=LSTM_HIDDEN,
                 mlp_hidden=512, dropout=0.3):
        super().__init__()
        seq_dim = 2 * rnn_hidden
        self.bert       = AutoModel.from_pretrained(bert_model_name)
        self.audio_enc  = ModalityEncoderRNN(audio_input_dim,  rnn_hidden)
        self.vision_enc = ModalityEncoderRNN(vision_input_dim, rnn_hidden)
        self.audio_attn  = CrossModalAttention(BERT_DIM, seq_dim)
        self.vision_attn = CrossModalAttention(BERT_DIM, seq_dim)
        self.text_attn_from_audio  = CrossModalAttention(seq_dim, BERT_DIM)
        self.text_attn_from_vision = CrossModalAttention(seq_dim, BERT_DIM)
        self.classifier = nn.Sequential(
            nn.Linear(BERT_DIM * 9, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 256),           nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
    def forward(self, input_ids, attention_mask, audio, vision):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        t        = bert_out.last_hidden_state[:, 0, :]   # (B, 768)
        text_seq = bert_out.last_hidden_state             # (B, L, 768)
        A = self.audio_enc(audio)
        V = self.vision_enc(vision)
        a_att, _    = self.audio_attn(t, A)
        v_att, _    = self.vision_attn(t, V)
        a_query     = A.mean(dim=1)
        v_query     = V.mean(dim=1)
        t_from_a, _ = self.text_attn_from_audio(a_query,  text_seq)
        t_from_v, _ = self.text_attn_from_vision(v_query, text_seq)
        h = torch.cat([t, a_att, v_att, t_from_a, t_from_v,
                        t - a_att, t - v_att,
                        t_from_a - t, t_from_v - t], dim=1)
        return self.classifier(h).squeeze(1)

In [19]:
torch.manual_seed(42)

# Exp 4: CMIA-RNN
print("=== Exp 4: CMIA-RNN — Alternating Training ===")
model_rnn = CMIARNNModel().to(DEVICE)
results["CMIA_RNN"] = train_alternating(
    model_rnn, train_loader, test_loader, label="CMIA_RNN")

# Exp 5: CMIA-Proj
print("=== Exp 5: CMIA-Proj — Alternating Training ===")
model_proj = CMIAProjModel().to(DEVICE)
proj_non_bert = (
    list(model_proj.text_proj.parameters())   +
    list(model_proj.audio_enc.parameters())   +
    list(model_proj.vision_enc.parameters())  +
    list(model_proj.audio_attn.parameters())  +
    list(model_proj.vision_attn.parameters()) +
    list(model_proj.classifier.parameters())
)
results["CMIA_Proj"] = train_alternating_generic(
    model_proj, proj_non_bert, train_loader, test_loader, label="CMIA_Proj")

# Exp 6: CMIA-Bidir
print("=== Exp 6: CMIA-Bidir — Alternating Training ===")
model_bidir = CMIABidirectionalModel().to(DEVICE)
results["CMIA_Bidir"] = train_alternating(
    model_bidir, train_loader, test_loader, label="CMIA_Bidir")

=== Exp 4: CMIA-RNN — Alternating Training ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_RNN] Ep 01 | Train Acc 0.5272 | Test Acc 0.4855  P 0.4538  R 1.0000  F1 0.6243  F1-mac 0.4041
  [CMIA_RNN] Ep 02 | Train Acc 0.5779 | Test Acc 0.6594  P 0.5652  R 0.8814  F1 0.6887  F1-mac 0.6564
  [CMIA_RNN] Ep 03 | Train Acc 0.6793 | Test Acc 0.6812  P 0.8000  R 0.3390  F1 0.4762  F1-mac 0.6235
  [CMIA_RNN] Ep 04 | Train Acc 0.7880 | Test Acc 0.7246  P 0.8387  R 0.4407  F1 0.5778  F1-mac 0.6867
  [CMIA_RNN] Ep 05 | Train Acc 0.8714 | Test Acc 0.7391  P 0.7949  R 0.5254  F1 0.6327  F1-mac 0.7152
  [CMIA_RNN] Ep 06 | Train Acc 0.8913 | Test Acc 0.7101  P 0.7436  R 0.4915  F1 0.5918  F1-mac 0.6836
  [CMIA_RNN] Ep 07 | Train Acc 0.9221 | Test Acc 0.6522  P 0.8235  R 0.2373  F1 0.3684  F1-mac 0.5642
  [CMIA_RNN] Ep 08 | Train Acc 0.9438 | Test Acc 0.7391  P 0.8108  R 0.5085  F1 0.6250  F1-mac 0.7125
  [CMIA_RNN] Ep 09 | Train Acc 0.9819 | Test Acc 0.6812  P 0.5882  R 0.8475  F1 0.6944  F1-mac 0.6806
  [CMIA_RNN] Ep 10 | Train Acc 0.9583 | Test Acc 0.6812  P 0.5843  R 0.8814  F1 0.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_Proj] Ep 01 | Train Acc 0.4656 | Test Acc 0.7101  P 0.7021  R 0.5593  F1 0.6226  F1-mac 0.6937
  [CMIA_Proj] Ep 02 | Train Acc 0.5996 | Test Acc 0.7319  P 0.6964  R 0.6610  F1 0.6783  F1-mac 0.7242
  [CMIA_Proj] Ep 03 | Train Acc 0.6649 | Test Acc 0.7391  P 0.7091  R 0.6610  F1 0.6842  F1-mac 0.7310
  [CMIA_Proj] Ep 04 | Train Acc 0.8025 | Test Acc 0.7609  P 0.7407  R 0.6780  F1 0.7080  F1-mac 0.7528
  [CMIA_Proj] Ep 05 | Train Acc 0.9094 | Test Acc 0.7391  P 0.6575  R 0.8136  F1 0.7273  F1-mac 0.7386
  [CMIA_Proj] Ep 06 | Train Acc 0.9221 | Test Acc 0.7174  P 0.6724  R 0.6610  F1 0.6667  F1-mac 0.7107
  [CMIA_Proj] Ep 07 | Train Acc 0.9511 | Test Acc 0.7319  P 0.7292  R 0.5932  F1 0.6542  F1-mac 0.7176
  [CMIA_Proj] Ep 08 | Train Acc 0.9837 | Test Acc 0.6957  P 0.6491  R 0.6271  F1 0.6379  F1-mac 0.6877
  [CMIA_Proj] Ep 09 | Train Acc 0.9982 | Test Acc 0.7101  P 0.6557  R 0.6780  F1 0.6667  F1-mac 0.7051
  [CMIA_Proj] Ep 10 | Train Acc 0.9946 | Test Acc 0.6957  P 0.6197  R 0.7

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_Bidir] Ep 01 | Train Acc 0.5725 | Test Acc 0.7246  P 0.6981  R 0.6271  F1 0.6607  F1-mac 0.7145
  [CMIA_Bidir] Ep 02 | Train Acc 0.5960 | Test Acc 0.6159  P 0.6667  R 0.2034  F1 0.3117  F1-mac 0.5227
  [CMIA_Bidir] Ep 03 | Train Acc 0.6739 | Test Acc 0.7101  P 0.7714  R 0.4576  F1 0.5745  F1-mac 0.6773
  [CMIA_Bidir] Ep 04 | Train Acc 0.7844 | Test Acc 0.7391  P 0.7255  R 0.6271  F1 0.6727  F1-mac 0.7279
  [CMIA_Bidir] Ep 05 | Train Acc 0.8678 | Test Acc 0.7391  P 0.7091  R 0.6610  F1 0.6842  F1-mac 0.7310
  [CMIA_Bidir] Ep 06 | Train Acc 0.8605 | Test Acc 0.7391  P 0.8286  R 0.4915  F1 0.6170  F1-mac 0.7096
  [CMIA_Bidir] Ep 07 | Train Acc 0.8967 | Test Acc 0.7464  P 0.6667  R 0.8136  F1 0.7328  F1-mac 0.7457
  [CMIA_Bidir] Ep 08 | Train Acc 0.9674 | Test Acc 0.7319  P 0.7115  R 0.6271  F1 0.6667  F1-mac 0.7212
  [CMIA_Bidir] Ep 09 | Train Acc 0.9783 | Test Acc 0.6884  P 0.6290  R 0.6610  F1 0.6446  F1-mac 0.6836
  [CMIA_Bidir] Ep 10 | Train Acc 0.9837 | Test Acc 0.7391  P 0.7

## 10. Final Results

The table below presents all CMIA variants alongside the reference baselines. Models are grouped by purpose: baselines, training strategy comparison, ablations, and improved variants.

**Columns**: *Precision* and *Recall* are computed for the positive (sarcastic) class.
*F1 (pos)* is the positive-class F1. *F1 (macro)* is the unweighted average of positive
and negative class F1, which is a fairer summary metric given MUStARD's class imbalance
(~60% non-sarcastic in the test set). A model that merely predicts the majority class
achieves ~0.72 accuracy but a macro F1 well below 0.70, so macro F1 is the primary
comparison metric.

**Reading the precision/recall columns**: Several CMIA variants achieve accuracy comparable to or exceeding late fusion, but the F1 ceiling remains similar. This is explained by a systematic precision-recall trade-off — models with higher accuracy tend to predict sarcasm more conservatively (higher precision, lower recall), while models with higher recall generate more false positives. F1 is the harmonic mean of both, so it stays bounded regardless of which direction the model leans. This behaviour reflects the inherent *text dominance* in MUStARD: the audio and vision features carry limited additional signal, so cross-modal incongruity features act as a soft corrective rather than a primary discriminator.

In [20]:
all_results = {**prior, **results}

final_order = [
    # Baselines
    ("Late Fusion RNN (avg, joint)", prior,   "Baseline — Late Fusion RNN (avg, joint)"),
    ("Early Fusion RNN (alt)",       prior,   "Baseline — Early Fusion RNN (alternating)"),
    ("BERT (ctx+utt, fine-tuned)",   prior,   "Baseline — BERT (ctx+utt, fine-tuned)"),
    # Exp 1: training strategy
    ("CMIA_joint",                   results, "Exp 1a — CMIA biLSTM, joint"),
    ("CMIA_alternating",             results, "Exp 1b — CMIA biLSTM, alternating"),
    # Exp 2–3: ablations
    ("CMIA_no_diff",                 results, "Exp 2  — CMIA, no diff terms (joint)"),
    ("CMIA_mean_pool",               results, "Exp 3  — CMIA, mean pool (joint)"),
    # Exp 4–6: improved variants
    ("CMIA_RNN",                     results, "Exp 4  — CMIA-RNN (alternating)"),
    ("CMIA_Proj",                    results, "Exp 5  — CMIA-Proj 256-dim (alternating)"),
    ("CMIA_Bidir",                   results, "Exp 6  — CMIA-Bidir (alternating)"),
]

def fmt(v): return f"{v:.4f}" if v is not None else "-"

def make_row(label, m):
    return {
        "Model / Variant": label,
        "Test Acc":   fmt(m.get("accuracy")),
        "Precision":  fmt(m.get("precision")),
        "Recall":     fmt(m.get("recall")),
        "F1 (pos)":   fmt(m.get("f1")),
        "F1 (macro)": fmt(m.get("f1_macro")),
    }

rows = [make_row(label, src.get(key, {})) for key, src, label in final_order]
display(pd.DataFrame(rows))

,Model / Variant,Test Acc,Precision,Recall,F1 (pos),F1 (macro)
0,"Baseline — Late Fusion RNN (avg, joint)",0.7899,0.7568,0.7458,0.7434,0.7909
1,Baseline — Early Fusion RNN (alternating),0.7464,-,-,0.7460,-
2,"Baseline — BERT (ctx+utt, fine-tuned)",0.7319,-,-,-,-
3,"Exp 1a — CMIA biLSTM, joint",0.7391,0.7091,0.6610,0.6842,0.7310
4,"Exp 1b — CMIA biLSTM, alternating",0.7681,0.7547,0.6780,0.7143,0.7596
5,"Exp 2 — CMIA, no diff terms (joint)",0.7319,0.6341,0.8814,0.7376,0.7318
6,"Exp 3 — CMIA, mean pool (joint)",0.7681,0.7547,0.6780,0.7143,0.7596
7,Exp 4 — CMIA-RNN (alternating),0.7391,0.7949,0.5254,0.6327,0.7152
8,Exp 5 — CMIA-Proj 256-dim (alternating),0.7609,0.7407,0.6780,0.7080,0.7528
9,Exp 6 — CMIA-Bidir (alternating),0.7464,0.6667,0.8136,0.7328,0.7457


## 11. Attention Weight Analysis (Optional)

To provide qualitative insight into what the cross-modal attention has learned, we extract the attention weight distributions for the best-performing CMIA model on a sample of test utterances. For each sample, we report the peak attention timestep and the entropy of the distribution: low entropy indicates sharp, focused attention (the model has identified a specific acoustic or visual moment as relevant); high entropy indicates diffuse attention across the sequence (consistent with audio/vision providing little discriminative signal).

In [ ]:
# Use the best CMIA model (alternating) for attention inspection
model_alt.eval()
sample_batch = next(iter(test_loader))

with torch.no_grad():
    ids   = sample_batch["input_ids"].to(DEVICE)
    mask  = sample_batch["attention_mask"].to(DEVICE)
    audio = sample_batch["audio"].to(DEVICE)
    vis   = sample_batch["vision"].to(DEVICE)
    lbls  = sample_batch["label"]

    bert_out = model_alt.bert(input_ids=ids, attention_mask=mask)
    t = bert_out.last_hidden_state[:, 0, :]
    A = model_alt.audio_enc(audio)
    V = model_alt.vision_enc(vis)
    _, audio_weights  = model_alt.audio_attn(t, A)   # (B, 50)
    _, vision_weights = model_alt.vision_attn(t, V)  # (B, 50)

print("Audio attention weights — first 5 test samples (50 timesteps each):")
for i in range(min(5, audio_weights.size(0))):
    w    = audio_weights[i].cpu().numpy()
    peak = w.argmax()
    ent  = -(w * np.log(w + 1e-9)).sum()
    print(f"  Sample {i} (label={int(lbls[i])}) — "
          f"peak at t={peak}, weight={w[peak]:.4f}, entropy={ent:.2f}")

print()
print("Vision attention weights — first 5 test samples:")
for i in range(min(5, vision_weights.size(0))):
    w    = vision_weights[i].cpu().numpy()
    peak = w.argmax()
    ent  = -(w * np.log(w + 1e-9)).sum()
    print(f"  Sample {i} (label={int(lbls[i])}) — "
          f"peak at t={peak}, weight={w[peak]:.4f}, entropy={ent:.2f}")